# Undiagnosed Hypertension and Diabetes Screening with NHANES
## Part 4: Sensitivity analyses and survey-weighted performance

Upload `nhanes_analytic.csv` (from Part 1) when asked. Runtime is about 5 to 10 minutes.

### Step 4.1: Setup, data and shared functions (same as Part 2)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost", "lightgbm", "interpret"], check=False)
import os, re, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from interpret.glassbox import ExplainableBoostingClassifier

warnings.filterwarnings("ignore")
SEED = 2026
np.random.seed(SEED)
OUT = "/content/part4_out" if os.path.isdir("/content") else "./part4_out"
os.makedirs(OUT, exist_ok=True)

CSV = "/content/nhanes_analytic.csv" if os.path.isdir("/content") else "nhanes_analytic.csv"
if not os.path.exists(CSV):
    from google.colab import files
    print("Upload nhanes_analytic.csv from Part 1")
    up = files.upload()
    CSV = "/content/" + list(up.keys())[0]
data = pd.read_csv(CSV)
print(data.shape)

### Step 4.1.2: Predictor sets
- **Diabetes, main model:** usable in every cycle, so it can be tested on 2021-2023
- **Diabetes, extended model:** adds family history and gestational diabetes (not asked in 2021-2023), evaluated on 2017-2020 only
- **Hypertension:** blood pressure is excluded because it defines the outcome

In [ ]:
data["hx_htn"] = data["told_htn"]                      # ever told high blood pressure (1/0)
data["hx_dm"] = (data["told_dm"] == 1).astype(float)   # ever told diabetes
data.loc[data["told_dm"].isna(), "hx_dm"] = np.nan
data["smoking"] = data["smoking"].fillna("missing")

NUM_DM = ["age", "bmi", "waist_cm", "height_cm", "whtr", "sbp_mean", "dbp_mean", "sedentary_min"]
BIN_DM = ["sex_female", "hx_htn", "cvd_history", "active_leisure"]
CAT_DM = ["smoking", "education"]
NUM_HTN = ["age", "bmi", "waist_cm", "height_cm", "whtr", "sedentary_min"]
BIN_HTN = ["sex_female", "hx_dm", "cvd_history", "active_leisure"]
CAT_HTN = ["smoking", "education"]
EXTRA_DM = ["famhx_dm", "gdm"]

TASKS = {
    "diabetes": {"pop": "dm_at_risk_pop", "y": "undx_dm", "num": NUM_DM, "bin": BIN_DM, "cat": CAT_DM,
                 "simple": ["age", "sex_female", "bmi"]},
    "hypertension": {"pop": "htn_at_risk_pop", "y": "undx_htn", "num": NUM_HTN, "bin": BIN_HTN, "cat": CAT_HTN,
                     "simple": ["age", "sex_female", "bmi"]},
}

def task_data(task):
    t = TASKS[task]
    d = data[(data[t["pop"]] == 1) & data[t["y"]].notna()].copy()
    d["education"] = d["education"].map(lambda v: "missing" if pd.isna(v) else str(int(v)))
    return d

for task in TASKS:
    d = task_data(task)
    print(task, d.groupby("split")[TASKS[task]["y"]].agg(["size", "sum", "mean"]).round(4).to_dict())

### Step 4.1.3: Models
Tree models handle missing values natively; logistic regression and EBM use median/mode imputation
fitted on the training data only. No resampling is used, because oversampling distorts calibration.

In [ ]:
def preprocessor(num, bin_, cat, scale=True):
    num_pipe = [("imp", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_pipe.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_pipe), num),
        ("bin", SimpleImputer(strategy="most_frequent"), bin_),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", drop="first"))]), cat),
    ])

def tree_matrix(d, num, bin_, cat):
    X = d[num + bin_].copy()
    for c in cat:
        X = pd.concat([X, pd.get_dummies(d[c], prefix=c, dtype=float)], axis=1)
    # XGBoost rejects [, ] and < in column names, so keep names to letters, digits and underscores
    X.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", str(c)) for c in X.columns]
    return X

def make_model(name, params, num, bin_, cat):
    if name == "LR":
        return Pipeline([("pre", preprocessor(num, bin_, cat)), ("m", LogisticRegression(max_iter=2000, C=params.get("C", 1.0)))])
    if name == "EBM":
        return Pipeline([("pre", preprocessor(num, bin_, cat, scale=False)),
                         ("m", ExplainableBoostingClassifier(random_state=SEED, interactions=params.get("interactions", 10)))])
    if name == "XGBoost":
        return XGBClassifier(n_estimators=params["n"], max_depth=params["depth"], learning_rate=params["lr"],
                             subsample=0.8, colsample_bytree=0.8, min_child_weight=5, eval_metric="logloss",
                             random_state=SEED, n_jobs=-1)
    if name == "LightGBM":
        return LGBMClassifier(n_estimators=params["n"], num_leaves=params["leaves"], learning_rate=params["lr"],
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, min_child_samples=30,
                              random_state=SEED, verbose=-1)

GRIDS = {
    "LR": [{"C": c} for c in [0.01, 0.1, 1.0]],
    "EBM": [{"interactions": i} for i in [0, 10]],
    "XGBoost": [{"n": n, "depth": dp, "lr": lr} for n in [200, 500] for dp in [2, 3, 4] for lr in [0.03]],
    "LightGBM": [{"n": n, "leaves": lv, "lr": lr} for n in [200, 500] for lv in [7, 15] for lr in [0.03]],
}
TREE = {"XGBoost", "LightGBM"}

def fit_predict(name, params, tr, te, num, bin_, cat, y):
    m = make_model(name, params, num, bin_, cat)
    if name in TREE:
        Xtr = tree_matrix(tr, num, bin_, cat)
        Xte = tree_matrix(te, num, bin_, cat).reindex(columns=Xtr.columns, fill_value=0)
        m.fit(Xtr, tr[y])
        return m, m.predict_proba(Xte)[:, 1]
    cols = num + bin_ + cat
    m.fit(tr[cols], tr[y])
    return m, m.predict_proba(te[cols])[:, 1]

### Step 4.1.4: Metrics

In [ ]:
def calib_int_slope(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    lp = np.log(p / (1 - p)).reshape(-1, 1)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp, y).coef_[0][0]
    # calibration-in-the-large: intercept with the linear predictor as offset
    import statsmodels.api as sm
    inter = sm.GLM(y, np.ones_like(lp), family=sm.families.Binomial(), offset=lp.ravel()).fit().params[0]
    return inter, slope

def boot_auc(y, p, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    aucs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        aucs.append(roc_auc_score(y[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

def threshold_for_sensitivity(y, p, target=0.80):
    fpr, tpr, thr = roc_curve(y, p)
    return thr[np.argmax(tpr >= target)]

def evaluate(y, p, thr):
    y = np.asarray(y)
    lo, hi = boot_auc(y, p)
    inter, slope = calib_int_slope(y, p)
    flag = p >= thr
    tp, fp = (flag & (y == 1)).sum(), (flag & (y == 0)).sum()
    fn, tn = (~flag & (y == 1)).sum(), (~flag & (y == 0)).sum()
    return {"AUC": roc_auc_score(y, p), "AUC_lo": lo, "AUC_hi": hi, "AUPRC": average_precision_score(y, p),
            "Brier": brier_score_loss(y, p), "Cal_intercept": inter, "Cal_slope": slope,
            "Sensitivity": tp / (tp + fn), "Specificity": tn / (tn + fp), "PPV": tp / max(tp + fp, 1),
            "Flagged_%": 100 * flag.mean()}

### Step 4.2: Sensitivity analyses
Each row refits logistic regression and XGBoost (settings from Part 2) on 2011-2020 and tests on 2021-2023.
- **Diabetes, HbA1c or fasting glucose:** fasting subsample only, outcome adds FPG >= 126 mg/dL
- **Hypertension at 130/80:** 2017 ACC/AHA threshold instead of 140/90
- **Diabetes without blood pressure:** a fully cuff-free model

In [ ]:
# Settings selected on 2017-2020 in Part 2
BEST = {("diabetes", "LR"): {"C": 1.0}, ("diabetes", "XGBoost"): {"n": 200, "depth": 2, "lr": 0.03},
        ("hypertension", "LR"): {"C": 0.1}, ("hypertension", "XGBoost"): {"n": 200, "depth": 3, "lr": 0.03}}
NUM_DM_NOBP = [c for c in NUM_DM if c not in ("sbp_mean", "dbp_mean")]

SCENARIOS = [
    ("Main: undiagnosed diabetes (HbA1c)", "diabetes", "undx_dm", NUM_DM),
    ("Diabetes: HbA1c or fasting glucose", "diabetes", "undx_dm_fpg", NUM_DM),
    ("Diabetes: no blood pressure inputs", "diabetes", "undx_dm", NUM_DM_NOBP),
    ("Main: undiagnosed hypertension (140/90)", "hypertension", "undx_htn", NUM_HTN),
    ("Hypertension: 130/80 threshold", "hypertension", "undx_htn_130", NUM_HTN),
]
sens_rows, SENS_PRED = [], {}
for label, task, ycol, num in SCENARIOS:
    t = TASKS[task]
    d = data[(data[t["pop"]] == 1) & data[ycol].notna()].copy()
    d["education"] = d["education"].map(lambda v: "missing" if pd.isna(v) else str(int(v)))
    tr, va, te = d[d.split == "train"], d[d.split == "validation"], d[d.split == "test"]
    trva = pd.concat([tr, va])
    for name in ["LR", "XGBoost"]:
        p = BEST[(task, name)]
        _, pv = fit_predict(name, p, tr, va, num, t["bin"], t["cat"], ycol)
        thr = threshold_for_sensitivity(va[ycol], pv)
        _, pt = fit_predict(name, p, trva, te, num, t["bin"], t["cat"], ycol)
        SENS_PRED[(label, name)] = (te, pt)
        sens_rows.append({"analysis": label, "model": name, "n_test": len(te), "cases_test": int(te[ycol].sum()),
                          "prevalence_%": round(100 * te[ycol].mean(), 2), **evaluate(te[ycol], pt, thr)})
        print(f"{label:42s} {name:8s} AUC {sens_rows[-1]['AUC']:.3f}")
sens = pd.DataFrame(sens_rows).round(3)
sens.to_csv(os.path.join(OUT, "sensitivity_analyses.csv"), index=False)
print(sens[["analysis", "model", "n_test", "cases_test", "prevalence_%", "AUC", "AUC_lo", "AUC_hi", "Cal_intercept", "Cal_slope", "Sensitivity", "Specificity"]].to_string(index=False))

### Step 4.3: Survey-weighted performance on 2021-2023
NHANES oversamples some groups. Weighting by the examination weights makes the results describe the
US adult population. Weighted AUC uses the weights as sample weights; O/E compares weighted observed cases
with weighted predicted cases.

In [ ]:
w_rows = []
for label, task, ycol, num in SCENARIOS:
    if not label.startswith("Main"):
        continue
    for name in ["LR", "XGBoost"]:
        te, pt = SENS_PRED[(label, name)]
        y, w = te[ycol].values, te["wt_pooled"].values
        ok = ~np.isnan(w)
        rng = np.random.default_rng(SEED); aucs = []
        for _ in range(500):
            i = rng.integers(0, ok.sum(), ok.sum())
            yy, pp, ww = y[ok][i], pt[ok][i], w[ok][i]
            if yy.min() != yy.max():
                aucs.append(roc_auc_score(yy, pp, sample_weight=ww))
        w_rows.append({"analysis": label, "model": name,
                       "AUC_unweighted": roc_auc_score(y, pt),
                       "AUC_weighted": roc_auc_score(y[ok], pt[ok], sample_weight=w[ok]),
                       "AUC_weighted_lo": np.percentile(aucs, 2.5), "AUC_weighted_hi": np.percentile(aucs, 97.5),
                       "Prevalence_weighted_%": 100 * np.average(y[ok], weights=w[ok]),
                       "Mean_predicted_weighted_%": 100 * np.average(pt[ok], weights=w[ok]),
                       "O/E_weighted": np.average(y[ok], weights=w[ok]) / np.average(pt[ok], weights=w[ok])})
wtab = pd.DataFrame(w_rows).round(3)
wtab.to_csv(os.path.join(OUT, "survey_weighted_performance.csv"), index=False)
print(wtab.to_string(index=False))

### Step 4.4: Save

In [ ]:
print("Saved to", OUT, os.listdir(OUT))
try:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/part4_results", "zip", OUT)
    files.download("/content/part4_results.zip")
except Exception:
    pass

### What to send back
`part4_results.zip`, or the printed tables from Steps 4.2 and 4.3.